# Session 4 · Colorectal Cancer & Table 1

**R for All of Us Bootcamp · Day 2, 1–3 p.m. ET**

You will find **colorectal cancer** (colon and rectal, not anal) in EHR diagnoses, join Sessions 1–3 into one analytic file, build a **Table 1**, and model the odds of colorectal cancer with logistic regression.

| Part | Time | What you do |
| --- | --- | --- |
| Demonstration | 1–2 | Colorectal cancer query and code check, analytic file, Table 1, small-cell check, logistic regression |
| Lab, Level 1 | 2–2:45 | Your own Table 1; change factor level order and referents; Table 1 stratified by other variables |
| Lab, Level 2 (boss) | if time | Table 1 by age at colorectal cancer diagnosis: under 40 vs 40–49 |


### How the game works

- Run each **checkpoint** cell after a task. A pass earns points; a miss prints a tip.
- Stuck? Run `hint("ID")` for a clue. Each hint costs 1 point.
- **Level 1** tasks are for everyone. **Level 2 (boss level)** tasks are optional extras for when you finish early.
- Run `scoreboard()` at any time. At the end of the lab, report your **score only** in the Zoom poll.
- Never paste data, counts or screenshots of participant-level data into the Zoom chat.

## R basics refresher

Session 1 has the full introduction. The short version:

- **Run a cell** with **Shift + Enter**. Run cells from top to bottom. `[*]` means the cell is still running.
- `#` starts a **comment**: R ignores the rest of the line. `# TODO` marks where you write code.
- `install.packages("name")` once (buying the textbook); `library(name)` every time you open the notebook (taking it off the shelf).
- **Red text is not always an error.** `Error` stops the cell; `Warning` and messages (such as the tidyverse "Conflicts" list) do not.
- `x <- value` stores a value in an object named `x`. Names are case-sensitive: `AGECAT` is not `agecat`.
- `c("a", "b")` combines values. Text needs quotes; numbers, `TRUE`, `FALSE`, `NA` and object names do not.
- `df$column` picks a column; `df[rows, columns]` picks rows and columns; `df[df$Age_Survey < 25, ]` keeps the rows where the condition is `TRUE`.
- `==` compares (one `=` is only for arguments); `!=` not equal; `&` and; `|` or; `!` not.
- `x %in% c("a", "b")` is `TRUE` when `x` is any of the listed values. View the documentation for `%in%` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/match.html)
- `NA` means missing. Test for it with `is.na(x)`, never `x == NA`. View the documentation for `is.na()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/NA.html)
- `table(x, useNA = "ifany")` counts each value, including `NA`. View the documentation for `table()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/table.html)
- `nrow(df)` counts rows. View the documentation for `nrow()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/nrow.html)

## Setup

Run these cells once each time you open the notebook. If the second cell stops with an error, run `00_SETUP_C2025Q4R6.ipynb` (Python kernel) first, then restart this kernel.

In [ ]:
library(tidyverse)
library(bigrquery)
options(scipen = 999)   # print long concept codes in full, never as 1.2e+15

In [ ]:
# Load the workspace variables written by 00_SETUP_C2025Q4R6.ipynb
env_file <- "/home/jupyter/Setting_Env_Variables_p2.R"
if (!file.exists(env_file)) {
    stop("Run 00_SETUP_C2025Q4R6.ipynb (Python kernel) first, then restart this kernel.")
}
source(env_file)

BILLING <- Sys.getenv("GOOGLE_CLOUD_PROJECT")   # your project: it pays for each query
CDR     <- Sys.getenv("WORKSPACE_CDR")          # the All of Us data, e.g. <project>.C2025Q4R6
BUCKET  <- Sys.getenv("WORKSPACE_BUCKET")       # your workspace bucket (gs://...)

if (any(c(BILLING, CDR, BUCKET) %in% c("", "NOT_FOUND"))) {
    stop("A workspace variable is missing. Re-run 00_SETUP_C2025Q4R6.ipynb, then restart this kernel.")
}

In [ ]:
# ---- Game helpers: run once, no need to read ----
score <- 0
earned <- c()
hints_used <- c()

check <- function(id, ok, points, success, tip) {
    ok <- tryCatch(isTRUE(ok), error = function(e) FALSE)
    if (ok) {
        if (!(id %in% earned)) { score <<- score + points; earned <<- c(earned, id) }
        cat(sprintf("PASS     [%s] %s  (+%d)   Score: %d\n", id, success, points, score))
    } else {
        cat(sprintf("NOT YET  [%s] %s\n", id, tip))
    }
    invisible(ok)
}

hints <- list(
    "4.A" = "Look for C21, 154.2, 154.3, or names with anal, anus, appendix, secondary or metastatic in the list.",
    "4.B" = "Pick 4 to 8 variables for my_vars and one grouping variable for my_strat (for example CANCERTYPE: colon vs rectal).",
    "4.C" = "relevel(EDU6CAT, ref = 'College Degree') moves one level to the front. For income, write all six levels in factor(INC6CAT, levels = c('150K or more', ...)).",
    "4.D" = "table1(analytic, vars = c(...), by = 'INS3CAT'). The by variable can't also be in vars.",
    "4.H" = "Filter to CANCERANY == 'Colorectal cancer', then make ONSET from AGEdxMINANY < 40."
)

hint <- function(id) {
    if (!(id %in% hints_used)) { score <<- score - 1; hints_used <<- c(hints_used, id) }
    cat(sprintf("HINT [%s] %s\n(-1 point)   Score: %d\n", id, hints[[id]], score))
}

scoreboard <- function() {
    cat(sprintf("Score: %d   Checkpoints passed: %s\n", score,
                if (length(earned)) paste(earned, collapse = ", ") else "none yet"))
}

# Checks used by the checkpoints
one_row_per_person <- function(df) nrow(df) > 0 && !any(duplicated(df$person_id))
only_values <- function(x, allowed) length(x) > 0 && all(na.omit(as.character(x)) %in% allowed)
in_range <- function(x, low, high) any(!is.na(x)) && all(na.omit(x) >= low & na.omit(x) <= high)

In [ ]:
# ---- Save to and load from your workspace bucket ----
save_to_bucket <- function(df, filename) {
    dir.create("data", showWarnings = FALSE)
    local_file <- file.path("data", filename)
    write_excel_csv(df, local_file)
    system(paste0("gsutil cp ", local_file, " ", BUCKET, "/data/"), intern = TRUE)
    message("Saved ", filename, " to ", BUCKET, "/data/")
}

load_from_bucket <- function(filename) {
    dir.create("data", showWarnings = FALSE)
    system(paste0("gsutil cp ", BUCKET, "/data/", filename, " data/"), intern = TRUE)
    read_csv(file.path("data", filename), show_col_types = FALSE)
}

## Study cohort: participants born in 1975 or later

Every query in this notebook keeps only participants **born in 1975 or later**. Here is why we define the cohort by year of birth.

**1. The data have a cutoff date.** CDR v9 (`C2025Q4R6`) "includes participant data with a cutoff date of January 1, 2025"
([CDR v9 release notes](https://support.researchallofus.org/hc/en-us/articles/50653909888788-Curated-Data-Repository-CDR-version-9-Release-Notes)). Data curated after that date are not in this release.
The cutoff is when curation stopped, not a promise that every person has records up to that day.
Cutoff dates for every release are in the [Data Dictionaries](https://support.researchallofus.org/hc/en-us/articles/360033200232-Data-Dictionaries-for-the-Curated-Data-Repositories-CDRs).

**2. Dates of birth are protected.** All of Us changes dates to protect participants' privacy
([Participant Privacy Protections](https://support.researchallofus.org/hc/en-us/articles/4552681983764-Participant-Privacy-Protections); [Data Curation Questions](https://support.researchallofus.org/hc/en-us/articles/35017475171348)):

| Tier | Dates of events (surveys, measurements, diagnoses) | Date of birth |
| --- | --- | --- |
| **Registered** (`R...` datasets) | Shifted backwards by a random 1 to 365 days, the same shift for every date of a participant | Shifted by the same amount |
| **Controlled** (`C...` datasets, used here) | Real dates | **Generalized to year of birth** |

In the Controlled Tier, the year of birth is real, but the month and day of `birth_datetime` are not the participant's.
(Session 1 shows which month and day are stored.)

**3. So we use the year, and only the year.** Born in 1975 or later means a participant was **at most 49 at any point in 2024**,
the last full year before the cutoff. Using one fixed birth year keeps the same people in all four notebooks, whenever the code is run.
A cutoff written as a full date (for example, born after 1975-01-01) would depend on a month and day that are not real.

**4. Ages are computed from years too.** Age at an event = the event's year minus the year of birth. This is the age the person
reaches during that calendar year: it equals their true age, or is one year more if their birthday comes later that year.
It is **never lower** than the true age. The next note explains why that matters.

**Why not subtract dates?** If we subtracted the stored `birth_datetime` from an event date, the answer would depend on the stored month and day.
If that stored day falls later in the year than a participant's real birthday, the computed age comes out too young,
by up to a year. All of Us enrolls adults **18 years or older** ([Join All of Us](https://az.joinallofus.org/en/get-started)),
yet a participant who enrolled shortly after turning 18 could then appear to be **17** at their first survey.
In the Registered Tier the birth date and event dates move by the same shift, so date differences keep the right age,
but the dates themselves no longer match the calendar, so a fixed calendar cutoff would still be off by up to a year.
Year arithmetic avoids both problems: an adult who was 18 at enrollment always comes out as 18 or older.

The filter runs inside BigQuery, so only these participants' rows ever leave Google's servers. That makes every query smaller and faster.

**New in the next cell**

- SQL a query language that BigQuery runs on Google's servers; only the result comes back to your notebook. `SELECT` picks columns, `FROM` names a table, `JOIN ... ON` links two tables on a shared column, `WHERE` keeps the rows that match, and `IN (...)` matches any value in a list or subquery.  
  View the documentation for SQL by clicking [**here.**](https://cloud.google.com/bigquery/docs/reference/standard-sql/query-syntax)
- `str_glue()` builds text and fills in anything in `{curly braces}` with the value of that R object. We use it to put `CDR`, `FIRST_BIRTH_YEAR` and `UNDER50` into SQL.  
  View the documentation for `str_glue()` by clicking [**here.**](https://stringr.tidyverse.org/reference/str_glue.html)

In [ ]:
CUTOFF <- "2025-01-01"          # CDR v9 data cutoff (release notes)
FIRST_BIRTH_YEAR <- 1975     # born 1975 or later: at most 49 at any point in 2024

# A subquery: the person_id of everyone born in FIRST_BIRTH_YEAR or later.
# year_of_birth is real in the Controlled Tier; the month and day of birth_datetime are not.
# Other queries use it as  WHERE person_id IN ({UNDER50})
UNDER50 <- str_glue("
    SELECT person_id
    FROM `{CDR}.person`
    WHERE year_of_birth >= {FIRST_BIRTH_YEAR}")

In [ ]:
# ---- Run a query once, keep the result in your bucket (run once, no need to read) ----
# First run: BigQuery runs the SQL and saves the result as CSV files in your bucket.
# Later runs: the saved files are read back, with no new query (faster, and no query cost).
# After you change a query, use refresh = TRUE to run it again.
bq_to_df <- function(sql, name, refresh = FALSE) {
    path  <- file.path(BUCKET, "bq_exports", name, paste0(name, "_*.csv"))
    files <- suppressWarnings(system2("gsutil", c("ls", path), stdout = TRUE, stderr = FALSE))
    if (refresh || length(files) == 0 || !is.null(attr(files, "status"))) {
        message("Running the query for ", name, " in BigQuery...")
        bq_table_save(bq_project_query(BILLING, sql), path, destination_format = "CSV")
        files <- system2("gsutil", c("ls", path), stdout = TRUE)
    } else {
        message("Reading saved results for ", name, " (refresh = TRUE re-runs the query)")
    }
    bind_rows(lapply(files, function(f)
        read_csv(pipe(paste("gsutil cat", f)), col_types = cols(.default = col_character()),
                 show_col_types = FALSE)))
}

## Table helpers (run once)

This session needs **no extra packages**: Table 1 and the regression tables are built with base R and the tidyverse,
which are already installed. (Packages such as gtsummary can fail to install on the Workbench because some of their
dependencies need to be compiled; these helpers avoid that.)

The next cell defines three functions. You don't need to read their code to use them:

| Function | What it returns |
| --- | --- |
| `table1(df, vars, by, cont)` | One row per variable and level: **n (%)** for categories (percent of each column), **mean (SD)** and **median [min, max]** for the numeric variables named in `cont`, an **Overall** column, one column per group in `by`, and a **p-value** per variable |
| `or_table(fit, data, vars)` | For a fitted logistic model: the **odds ratio (95% CI)** for each level, **Ref** for the referent, and one **p-value per variable** |
| `uv_regression(data, vars, outcome)` | One logistic model per variable (univariable), stacked with `or_table()` |

**How the numbers are made**

- **Categorical p-values:** chi-squared test of the variable by group (`chisq.test()`). When any expected cell count is under 5,
  the p-value is simulated from 10,000 random tables instead (`simulate.p.value = TRUE`), as our gtsummary template did with Fisher's test.  
  View the documentation for `chisq.test()` by clicking [**here.**](https://search.r-project.org/R/refmans/stats/html/chisq.test.html)
- **Numeric p-values:** Kruskal–Wallis rank-sum test (`kruskal.test()`), which compares groups without assuming a normal distribution
  (with two groups it is the same as the Wilcoxon rank-sum test).  
  View the documentation for `kruskal.test()` by clicking [**here.**](https://search.r-project.org/R/refmans/stats/html/kruskal.test.html)
- **Odds ratios:** `exp()` of each model coefficient; the 95% CI is `exp(estimate ± 1.96 × SE)` (Wald interval, `confint.default()`).  
  View the documentation for `confint.default()` by clicking [**here.**](https://search.r-project.org/R/refmans/stats/html/confint.html)
- **One p-value per variable:** a likelihood ratio test that drops the whole variable from the model and asks whether the fit gets worse
  (`drop1(fit, test = "LRT")`). This replaces `add_global_p()`, without the car package.  
  View the documentation for `drop1()` by clicking [**here.**](https://search.r-project.org/R/refmans/stats/html/add1.html)
- Missing values (`NA`) are shown as an **Unknown** row and left out of the p-value, as gtsummary does.
- p-values are shown to 3 decimals, and as `<0.001` when smaller.

The tables are ordinary tibbles, so Jupyter shows them as formatted tables, and you can save one with `write_csv()`.

In [ ]:
# ---- Table helpers: run once (base R + tidyverse only, no extra packages) ----
options(repr.matrix.max.rows = 500, repr.matrix.max.cols = 50)   # show whole tables in Jupyter

fmt_p <- function(p) ifelse(is.na(p), "", ifelse(p < 0.001, "<0.001", sprintf("%.3f", p)))
fmt_n <- function(n) format(n, big.mark = ",", trim = TRUE)

# Table 1: n (%) for categories, mean (SD) and median [min, max] for numbers, by group, plus Overall and a p-value
table1 <- function(df, vars, by, cont = character(0)) {
    df <- df[!is.na(df[[by]]), ]
    g  <- factor(df[[by]])
    cols <- c("Overall", levels(g))
    split_x <- function(x) c(list(Overall = x), split(x, g))
    line <- function(variable, level, values, p = "") {
        out <- tibble(Variable = variable, Level = level)
        for (k in seq_along(cols)) out[[cols[k]]] <- values[k]
        out$p <- p
        out
    }
    rows <- list(line("N", "", fmt_n(c(nrow(df), as.vector(table(g))))))
    for (v in vars) {
        x <- df[[v]]
        if (v %in% cont) {
            parts <- split_x(as.numeric(x))
            p <- tryCatch(kruskal.test(as.numeric(x) ~ g)$p.value, error = function(e) NA)
            rows <- c(rows, list(
                line(v, "", rep("", length(cols)), fmt_p(p)),
                line("", "Mean (SD)", sapply(parts, function(z) sprintf("%.1f (%.1f)", mean(z, na.rm = TRUE), sd(z, na.rm = TRUE)))),
                line("", "Median [min, max]", sapply(parts, function(z) sprintf("%.1f [%.1f, %.1f]", median(z, na.rm = TRUE),
                                                                                 min(z, na.rm = TRUE), max(z, na.rm = TRUE))))))
        } else {
            x   <- if (is.factor(x)) x else factor(x)
            tt  <- table(x, g)                                     # for the test: known values only
            tt  <- tt[rowSums(tt) > 0, , drop = FALSE]            # empty levels can't be tested
            if (any(is.na(x))) {                                   # show missing values as "Unknown"
                x <- factor(ifelse(is.na(x), "Unknown", as.character(x)), levels = c(levels(x), "Unknown"))
            }
            tab <- cbind(Overall = table(x), table(x, g))          # counts: one row per level, one column per group
            pct <- sweep(tab, 2, colSums(tab), "/") * 100          # column percentages
            p <- NA
            if (nrow(tt) > 1 && ncol(tt) > 1) {
                expected <- suppressWarnings(chisq.test(tt)$expected)
                p <- if (any(expected < 5)) chisq.test(tt, simulate.p.value = TRUE, B = 10000)$p.value
                     else chisq.test(tt)$p.value
            }
            rows <- c(rows, list(line(v, "", rep("", length(cols)), fmt_p(p))))
            for (lv in rownames(tab)) {
                rows <- c(rows, list(line("", lv, sprintf("%s (%.1f%%)", fmt_n(tab[lv, ]), pct[lv, ]))))
            }
        }
    }
    bind_rows(rows)
}

# Odds ratios (95% CI) for each level of each variable in a fitted logistic model, plus one p-value per variable
or_table <- function(fit, data, vars) {
    est <- coef(summary(fit))
    ci  <- suppressMessages(confint.default(fit))                 # Wald 95% CI: estimate +/- 1.96 x SE
    lrt <- drop1(fit, test = "LRT")                               # likelihood ratio test for each variable
    bind_rows(lapply(vars, function(v) {
        lv   <- levels(data[[v]])
        term <- paste0(v, lv)
        or <- sapply(seq_along(lv), function(i) {
            if (i == 1) return("Ref")
            if (!term[i] %in% rownames(est) || is.na(coef(fit)[term[i]])) return("—")
            sprintf("%.2f (%.2f, %.2f)", exp(est[term[i], 1]), exp(ci[term[i], 1]), exp(ci[term[i], 2]))
        })
        bind_rows(tibble(Variable = v, Level = "", OR = "", p = fmt_p(lrt[v, "Pr(>Chi)"])),
                  tibble(Variable = "", Level = lv, OR = or, p = ""))
    }))
}

# Univariable: one logistic model per variable, stacked
uv_regression <- function(data, vars, outcome) {
    bind_rows(lapply(vars, function(v) {
        fit <- glm(reformulate(v, response = outcome), data = data, family = "binomial")
        or_table(fit, data, v)
    }))
}

# Part A · Demonstration

## A1. Load your files from Sessions 1–3

In [ ]:
demodf <- load_from_bucket("S1_DEMOGRAPHICS.csv")
ehhwb  <- load_from_bucket("S2_PHQ_GAD.csv")
bmi    <- load_from_bucket("S3_BMI.csv")

c(demographics = nrow(demodf), phq_gad = nrow(ehhwb), bmi = nrow(bmi))

## A1b. Set the factor levels again

In Session 1 we made the category columns into **factors**, so they print in a set order (lowest to highest, Missing last).
A **CSV file stores only plain text**: it keeps the values (`"50-74K"`) but not the fact that the column is a factor,
and not the order of its levels. When `read_csv()` reads the file back, every category column comes back as **character**,
sorted alphabetically again.

So each time you load a CSV, convert the columns to factors again, with the same levels in the same order.
(An `.rds` file, written with `saveRDS()` and read with `readRDS()`, does keep factors, but only R can open it. CSV can be opened anywhere.)

**New in the next cell**

- `factor(x, levels = c(...))` turns a text column into a **factor**: R's type for categories, with a fixed list of allowed values (**levels**) in the order you choose. See the explanation just above.  
  View the documentation for `factor()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/factor.html)

In [ ]:
# After loading: character, not factor
class(demodf$INC6CAT)

age_levels  <- c("18-24", "25-34", "35-49")
race_levels <- c("Hispanic/Latino-a-e", "NH Black", "NH White", "NH Other", "Missing race/ethnicity")
edu_levels  <- c("Less than HS/GED", "HS/GED", "Some College", "College Degree", "Advanced Degree", "Missing education")
inc_levels  <- c("Less than 50K", "50-74K", "75-99K", "100-149K", "150K or more", "Missing income")
ins_levels  <- c("Insurance", "No insurance", "Missing insurance")
sab_levels  <- c("Male", "Female", "Other", "Missing sex")

demodf$AGECAT   <- factor(demodf$AGECAT,   levels = age_levels)
demodf$RACE5CAT <- factor(demodf$RACE5CAT, levels = race_levels)
demodf$EDU6CAT  <- factor(demodf$EDU6CAT,  levels = edu_levels)
demodf$INC6CAT  <- factor(demodf$INC6CAT,  levels = inc_levels)
demodf$INS3CAT  <- factor(demodf$INS3CAT,  levels = ins_levels)
if ("SABCAT" %in% names(demodf)) {          # only if you finished the Session 1 boss level
    demodf$SABCAT <- factor(demodf$SABCAT, levels = sab_levels)
}

# Factor again, in our order.
# Count NA in each column. AGECAT is NA only for anyone with no survey date (no age); the others should be 0.
# A new NA after factor() means a label in the data doesn't match a level exactly.
class(demodf$INC6CAT)
levels(demodf$INC6CAT)
colSums(is.na(demodf[, c("AGECAT", "RACE5CAT", "EDU6CAT", "INC6CAT", "INS3CAT")]))

## A2. Get colorectal cancer diagnoses from the EHR

Diagnoses are found two ways. **ICD codes** (what most clinics record): colon (C18, 153), rectosigmoid junction (C19, 154.0) and rectum (C20, 154.1). **SNOMED codes**: the SNOMED concepts those ICD codes map to, plus all their more specific children, for clinics that record SNOMED only. Anal cancers, the appendix, and cancer that spread to the colon from elsewhere are left out.

The query uses `WITH` to build three small lists first (the ICD codes, their SNOMED concepts, and the final SNOMED list), then uses them to pick the diagnoses.

**New in the next cell**

- `%>%` (the pipe) passes the result on its left into the function on its right, so steps read top to bottom. Read it as "and then": `df %>% filter(...) %>% mutate(...)`.  
  View the documentation for `%>%` by clicking [**here.**](https://magrittr.tidyverse.org/reference/pipe.html)
- `mutate()` adds a new column or changes an existing one. Every row is kept.  
  View the documentation for `mutate()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/mutate.html)

In [ ]:
# Colorectal cancer (colon, rectosigmoid junction, rectum), under-50 cohort, found two ways:
#  1. by ICD code: ICD-10-CM C18, C19, C20; ICD-9-CM 153, 154.0, 154.1
#  2. by SNOMED: the SNOMED concepts those ICD codes map to, and all their children
#     (catches diagnoses that a clinic recorded with a SNOMED code only)
# Left out: anal cancer (C21; 154.2, 154.3), the appendix (C18.1; 153.5), which cancer registries
# count separately from colon, and cancer that spread to the colon from elsewhere (C78.5; 197.5).
# The SNOMED concepts for those left-out codes, and their children, are removed from part 2.
crc_sql <- str_glue("
    WITH icd AS (
        SELECT concept_id,
               CASE
                   WHEN (vocabulary_id = 'ICD10CM' AND REGEXP_CONTAINS(concept_code, r'^C(18|19|20)')
                         AND concept_code != 'C18.1')
                     OR (vocabulary_id = 'ICD9CM' AND ((STARTS_WITH(concept_code, '153') AND concept_code != '153.5')
                                                       OR concept_code IN ('154.0', '154.1')))
                   THEN 'include'
                   WHEN concept_code = '154.8' THEN 'neither'    -- rectum and anus together: not used either way
                   ELSE 'exclude'
               END AS grp
        FROM `{CDR}.concept`
        WHERE (vocabulary_id = 'ICD10CM' AND REGEXP_CONTAINS(concept_code, r'^(C18|C19|C20|C21|C78\\.5)'))
           OR (vocabulary_id = 'ICD9CM'  AND REGEXP_CONTAINS(concept_code, r'^(153|154|197\\.5)'))
    ),
    snomed AS (
        SELECT ca.descendant_concept_id AS concept_id, icd.grp
        FROM icd
        JOIN `{CDR}.concept_relationship` cr
          ON cr.concept_id_1 = icd.concept_id AND cr.relationship_id = 'Maps to'
        JOIN `{CDR}.concept_ancestor` ca
          ON ca.ancestor_concept_id = cr.concept_id_2
    ),
    crc_snomed AS (
        SELECT concept_id FROM snomed WHERE grp = 'include'
        EXCEPT DISTINCT
        SELECT concept_id FROM snomed WHERE grp = 'exclude'
    )
    SELECT co.person_id,
           co.condition_start_datetime,
           src.vocabulary_id AS source_vocabulary,
           src.concept_code  AS source_concept_code,
           std.concept_code  AS standard_concept_code,
           std.concept_name  AS standard_concept_name
    FROM `{CDR}.condition_occurrence` co
    LEFT JOIN `{CDR}.concept` src ON co.condition_source_concept_id = src.concept_id
    LEFT JOIN `{CDR}.concept` std ON co.condition_concept_id = std.concept_id
    WHERE co.person_id IN ({UNDER50})
      AND (co.condition_source_concept_id IN (SELECT concept_id FROM icd WHERE grp = 'include')
           OR co.condition_concept_id IN (SELECT concept_id FROM crc_snomed))")

neo1df <- bq_to_df(crc_sql, "s4_crc_snomed_yob1975") %>%
    mutate(person_id = as.numeric(person_id))

n_distinct(neo1df$person_id)

## A3. Query check: which codes came back?

Always look at the codes a query returns, especially after a change. This lists each code and SNOMED name once (no counts), then checks that no anal, appendix or secondary cancer slipped in.

**New in the next cell**

- `distinct()` removes duplicate rows. With column names, it keeps one row per combination of those columns (`.keep_all = TRUE` keeps the other columns too).  
  View the documentation for `distinct()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/distinct.html)
- `arrange()` sorts the rows. `desc()` sorts largest or newest first.  
  View the documentation for `arrange()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/arrange.html)
- `substr(x, start, stop)` keeps characters `start` to `stop` of a text value: `substr("2023-08-01 09:00:00", 1, 10)` gives `"2023-08-01"`.  
  View the documentation for `substr()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/substr.html)

In [ ]:
neo1df %>%
    distinct(source_vocabulary, source_concept_code, standard_concept_name) %>%
    arrange(source_vocabulary, source_concept_code)

In [ ]:
anal_codes <- grepl("^C21|^154\\.[238]", neo1df$source_concept_code)
left_out  <- grepl("\\banal\\b|\\banus\\b|appendix|secondary|metasta", neo1df$standard_concept_name, ignore.case = TRUE)
check("4.A", nrow(neo1df) > 0 && !any(anal_codes) && !any(left_out),
      5, "Only colon, rectosigmoid and rectal cancers: no anal, appendix or secondary cancers.",
      "A left-out code or name came back. Look at the list above, then check the query.")

## A4. Colon or rectal, for each diagnosis

**Age at diagnosis = diagnosis year − year of birth**, for the same reason as age at survey in Session 1: in the Controlled Tier only the year of birth is real ([Participant Privacy Protections](https://support.researchallofus.org/hc/en-us/articles/4552681983764-Participant-Privacy-Protections)). The result is the true age at diagnosis or one year more, never less. Keep this in mind at a cut point: someone diagnosed at 39, before their 40th birthday that year, is counted as 40 (the 40–49 group in Task 4.H).

EHR records can include care from before a participant enrolled, so an age at diagnosis under 18 can be real here, unlike age at The Basics survey, which is taken at enrollment.

**New in the next cell**

- `select()` keeps only the columns you name, in the order you name them.  
  View the documentation for `select()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/select.html)
- `left_join(x, y, by = "person_id")` keeps every row of `x` and adds the matching columns from `y`. People with no match in `y` get `NA`.  
  View the documentation for `left_join()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/mutate-joins.html)
- `as.Date()` turns text such as `"2023-08-01"` into a date that R can sort, compare and subtract. `format = "%Y-%m-%d"` tells R the layout (year-month-day).  
  View the documentation for `as.Date()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/as.Date.html)
- `format(x, "%Y")` writes a date as text in the layout you give; `"%Y"` is the 4-digit year. `as.numeric(format(SURVdt, "%Y"))` turns 2023-08-01 into the number 2023.  
  View the documentation for `format()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/format.Date.html)
- `case_when()` a list of `condition ~ value` rules, checked in order; each row gets the value of the first rule that is `TRUE`. `.default` is the value when no rule matches.  
  View the documentation for `case_when()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/case_when.html)

In [ ]:
# Year of birth from Session 1 (the month and day of birth are not real in the Controlled Tier)
final0 <- demodf %>%
    select(person_id, year_of_birth) %>%
    mutate(year_of_birth = as.numeric(year_of_birth))

neo2df <- neo1df %>%
    left_join(final0, by = "person_id") %>%
    mutate(
        NEOdt = as.Date(substr(condition_start_datetime, 1, 10), format = "%Y-%m-%d"),
        # Age reached in the diagnosis year: diagnosis year minus year of birth (true age, or one more)
        AGEdx = as.numeric(format(NEOdt, "%Y")) - year_of_birth,
        # ICD codes decide first: C18 and 153 are colon; C19, C20, 154.0 and 154.1 are rectal.
        # SNOMED-only diagnoses: a name mentioning the rectum (or rectosigmoid) is rectal, the rest colon.
        CRCSITE = case_when(
            substr(source_concept_code, 1, 3) %in% c("C18", "153") ~ "Colon",
            substr(source_concept_code, 1, 3) %in% c("C19", "C20", "154") ~ "Rectal",
            grepl("rect", standard_concept_name, ignore.case = TRUE) ~ "Rectal",
            .default = "Colon"))

table(neo2df$CRCSITE, useNA = "ifany")

## A5. One row per person

**New in the next cell**

- `group_by()` splits the rows into groups (here, one group per person) so the next step works within each group.  
  View the documentation for `group_by()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/group_by.html)
- `summarise()` / `summarize()` collapses each group to one row, computing values such as `n()` (row count), `mean()` or `max()`.  
  View the documentation for `summarise()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/summarise.html)
- `ifelse(test, yes, no)` gives `yes` where the test is `TRUE` and `no` where it is `FALSE`, row by row.  
  View the documentation for `ifelse()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/ifelse.html)

In [ ]:
# One row per person with colorectal cancer: site, first diagnosis, age at first diagnosis
neo_person <- neo2df %>%
    group_by(person_id) %>%
    summarise(
        AGEdxMINANY = suppressWarnings(min(AGEdx, na.rm = TRUE)),
        COLON       = as.numeric(any(CRCSITE == "Colon")),
        RECTAL      = as.numeric(any(CRCSITE == "Rectal")),
        COUNTNEOANY = n_distinct(NEOdt),
        NEOANYdtMIN = suppressWarnings(min(NEOdt, na.rm = TRUE))) %>%
    mutate(
        AGEdxMINANY = ifelse(is.infinite(AGEdxMINANY), NA, AGEdxMINANY),
        CANCERTYPE = case_when(
            COLON == 1 & RECTAL == 1 ~ "Colon and rectal",
            COLON == 1 ~ "Colon",
            RECTAL == 1 ~ "Rectal"))

nrow(neo_person)
any(duplicated(neo_person$person_id))
table(neo_person$CANCERTYPE, useNA = "ifany")

## A6. Analytic file: everyone, with or without colorectal cancer

**New in the next cell**

- `replace_na()` replaces `NA` with the value you give, e.g. `replace_na(INC6CAT, "Missing income")`.  
  View the documentation for `replace_na()` by clicking [**here.**](https://tidyr.tidyverse.org/reference/replace_na.html)

In [ ]:
analytic <- demodf %>%
    left_join(neo_person %>% select(person_id, CANCERTYPE, AGEdxMINANY, NEOANYdtMIN), by = "person_id") %>%
    left_join(ehhwb %>% select(person_id, PHQCAT, GADCAT), by = "person_id") %>%
    left_join(bmi %>% select(person_id, BMICAT), by = "person_id") %>%
    mutate(
        CANCERANY = ifelse(is.na(CANCERTYPE), "No colorectal cancer", "Colorectal cancer"),
        PHQCAT = replace_na(PHQCAT, "Missing PHQ"),
        GADCAT = replace_na(GADCAT, "Missing GAD"),
        BMICAT = replace_na(BMICAT, "Missing BMI"))

# These three came from CSV files too, so set their factor levels here (after the NA labels are added)
phq_levels <- c("Minimal", "Mild", "Moderate", "Moderately severe", "Severe", "Missing PHQ")
gad_levels <- c("Minimal", "Mild", "Moderate", "Severe", "Missing GAD")
bmi_levels <- c("Underweight", "Normal", "Overweight", "Obese", "Missing BMI")

analytic$PHQCAT <- factor(analytic$PHQCAT, levels = phq_levels)
analytic$GADCAT <- factor(analytic$GADCAT, levels = gad_levels)
analytic$BMICAT <- factor(analytic$BMICAT, levels = bmi_levels)

colSums(is.na(analytic[, c("PHQCAT", "GADCAT", "BMICAT")]))   # should all be 0

nrow(analytic)
any(duplicated(analytic$person_id))

## A7. Table 1

Rows are variables and levels; columns are **Overall** and each cancer group, each cell **n (%)** of that column, with one **p-value** per variable. `Age_Survey` shows mean (SD) and median [min, max].

In [ ]:
# Table 1 (template: GTSUMMARY TABLE 1), built with our table1() helper
allVars  <- c("Age_Survey", "AGECAT", "RACE5CAT", "EDU6CAT", "INC6CAT", "INS3CAT", "PHQCAT", "GADCAT", "BMICAT")
contVars <- c("Age_Survey")
factVars <- c("AGECAT", "RACE5CAT", "EDU6CAT", "INC6CAT", "INS3CAT", "PHQCAT", "GADCAT", "BMICAT")
stratVar <- "CANCERANY"

# Sex at birth, if you finished the Session 1 boss level: all four groups, including Other and Missing sex
if ("SABCAT" %in% names(analytic)) {
    allVars  <- append(allVars, "SABCAT", after = 2)
    factVars <- append(factVars, "SABCAT", after = 1)
}

tableU <- table1(analytic, vars = allVars, by = stratVar, cont = contVars)
tableU

## A8. Before you share: no group under 20

All of Us does not allow sharing any count from 1 to 19, or a number someone could use to work one out. This check lists the groups to collapse. An empty result means the table is safe to share.

**New in the next cell**

- `filter()` keeps only the rows where the condition is `TRUE`.  
  View the documentation for `filter()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/filter.html)

In [ ]:
# Small-cell check: list any group with 1 to 19 people (levels only, no counts printed)
small_cells <- function(df, vars, by) {
    bind_rows(lapply(vars, function(v) {
        by_group <- df %>% count(level = as.character(.data[[v]]), group = as.character(.data[[by]]))
        overall  <- df %>% count(level = as.character(.data[[v]])) %>% mutate(group = "Overall")
        bind_rows(by_group, overall) %>%
            filter(n > 0, n < 20) %>%
            transmute(variable = v, level, group)
    }))
}

small_cells(analytic, factVars, stratVar)

## A9. Odds of colorectal cancer: univariable and multivariable logistic regression

**The outcome.** Logistic regression needs a 0/1 outcome: `CANCER01` is 1 for colorectal cancer and 0 for everyone else.

**The referent (reference group).** For a factor, R compares every level to the **first level**, the referent.
Each odds ratio (OR) in the table reads "the odds of colorectal cancer in this group compared with the referent". The referent itself shows a dash (—).
Because we set `levels` in a deliberate order, **the first value in each `levels = c(...)` list is the referent**:

| Variable | Referent (first level) |
| --- | --- |
| `AGECAT` | 18-24 |
| `RACE5CAT` | Hispanic/Latino-a-e, **changed to NH White below** |
| `EDU6CAT` | Less than HS/GED |
| `INC6CAT` | Less than 50K |
| `INS3CAT` | Insurance |
| `SABCAT` (if you finished the Session 1 boss level) | Male (only Male and Female stay in the models, see below) |
| `PHQCAT`, `GADCAT` | Minimal |
| `BMICAT` | Underweight, **changed to Normal below** |

**Sex at birth in the models: Male and Female only.** Table 1 shows all four sex-at-birth groups. Before the models, we remove
everyone whose `SABCAT` is **"Other"** or **"Missing sex"**:

- **"Other":** in this cohort, Table 1 shows **0** participants with "Other" sex at birth who have colorectal cancer.
  With no one in a group having the outcome, logistic regression cannot estimate that group's odds ratio: the estimate keeps
  moving toward 0 while its standard error grows without limit, so the 95% confidence interval runs from about 0 to an enormous
  number. That is not an acceptable estimate, and it can make the rest of the model unstable. (Statisticians call this
  *separation*.) Combining "Other" with another group would not be meaningful here.
- **"Missing sex":** this is not a sex category, so an odds ratio for it would have no clear meaning.

So the models describe participants whose sex at birth is Male or Female, and they include fewer people than Table 1.
Say so in your methods. (For sparse data like this, penalized methods such as Firth's logistic regression are another option,
beyond today's session.)

**Choosing a referent.** The order that reads best in a table is not always the best comparison group for a model. A good referent is:

- **meaningful**: the comparison answers the question you are asking, and
- **large**: the referent's count sits under every odds ratio for that variable, so a small referent makes every OR in the variable unstable, with wide confidence intervals.

For BMI, **Normal** is the usual referent: it is the clinical reference point, and Underweight is a small group.
For race and ethnicity, many studies use **NH White**, often the largest group, so each OR compares a group with it.
(Choosing a referent is a decision to state in your methods; it changes what each OR compares, not the model's fit.)

**Changing the referent with `relevel()`.** `relevel(x, ref = "Normal")` moves one level to the front, making it the referent, and keeps the
other levels in their order. You could instead rewrite the whole `levels = c(...)` list with the referent first; `relevel()` is shorter.

We change the referents in a **copy** of the data, `regdf`, used only for the models. `analytic` keeps the table order
(Underweight first, Hispanic/Latino-a-e first), so Table 1 and your own tables in Part B still read in that order.
`levels(regdf$BMICAT)[1]` shows the current referent.
A text (character) column also works in `glm()`, but then the referent is simply whichever value comes first alphabetically. That is one more reason to set factor levels yourself.

**The code** follows our regression template (univariable and multivariable side by side), using the helpers above:

1. `uv_regression()` fits one logistic model **per variable** (univariable: each variable alone) with `glm(..., family = "binomial")`,
   and turns each into odds ratios with `or_table()`.
2. `glm()` fits **one model with all the variables together** (multivariable: each OR is adjusted for the others).
   `summary()` shows the raw model, where coefficients are log-odds; `or_table()` turns them into **odds ratios** with 95% CIs.
3. `bind_cols()` puts the two tables side by side: **Univariable** columns, then **Multivariable** columns.
4. Each variable gets **one p-value** (likelihood ratio test from `drop1()`): does the variable as a whole improve the model?

The **Missing** groups stay in the models as their own levels, so no one is dropped.
A level with very few people, or with no one who has the outcome, gives a huge or near-zero OR with a very wide confidence interval. If you see that, combine it with a neighbouring level or choose a larger referent. `droplevels()` removes any level with no people at all, so it never appears in the models. A person with a missing (`NA`) value in a model's variables is left out of that model; here that is only `AGECAT` for anyone with no survey date.

**New in the next cell**

- `relevel(x, ref = "...")` moves one level of a factor to the front, making it the referent; the other levels keep their order.  
  View the documentation for `relevel()` by clicking [**here.**](https://search.r-project.org/R/refmans/stats/html/relevel.html)
- `droplevels()` removes factor levels that no row uses. On a whole dataframe it does this for every factor column.  
  View the documentation for `droplevels()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/droplevels.html)
- `glm()` fits a generalized linear model. With `family = "binomial"` and a 0/1 outcome it is **logistic regression**. The formula `y ~ a + b` reads "y modelled by a and b".  
  View the documentation for `glm()` by clicking [**here.**](https://search.r-project.org/R/refmans/stats/html/glm.html)
- `summary()` a quick look at a numeric column: minimum, quartiles, median, mean, maximum and the number of NAs.  
  View the documentation for `summary()` by clicking [**here.**](https://search.r-project.org/R/refmans/base/html/summary.html)
- `bind_cols()` puts tables side by side, row 1 next to row 1. The tables must have the same rows in the same order.  
  View the documentation for `bind_cols()` by clicking [**here.**](https://dplyr.tidyverse.org/reference/bind_cols.html)

In [ ]:
# Outcome: colorectal cancer, 1 = yes, 0 = no
analytic <- analytic %>%
    mutate(CANCER01 = ifelse(CANCERANY == "Colorectal cancer", 1, 0))
table(analytic$CANCER01, useNA = "ifany")

# Sex at birth in the models: Male and Female only (see the explanation above).
# Table 1 above still shows all four groups.
regbase <- analytic
if ("SABCAT" %in% names(regbase)) {
    regbase <- regbase %>% filter(!SABCAT %in% c("Other", "Missing sex"))
    print(table(regbase$SABCAT, regbase$CANCERANY))   # Other and Missing sex now have 0 rows
}

# A copy for the models, with the referents we want:
#   BMI: Normal (the clinical reference point; Underweight is a small group)
#   Race and ethnicity: NH White
# droplevels() removes any level with no one in it (here: Other and Missing sex), so every level in the models has data
regdf <- regbase %>%
    mutate(BMICAT   = relevel(BMICAT,   ref = "Normal"),
           RACE5CAT = relevel(RACE5CAT, ref = "NH White")) %>%
    droplevels()

levels(analytic$BMICAT)   # tables: Underweight first
levels(regdf$BMICAT)      # models: Normal first, the rest in the same order
levels(regdf$RACE5CAT)

# The predictors (SABCAT only if you finished the Session 1 boss level)
predVars <- c("AGECAT", "RACE5CAT", "EDU6CAT", "INC6CAT", "INS3CAT", "PHQCAT", "GADCAT", "BMICAT")
if ("SABCAT" %in% names(regdf)) predVars <- c("AGECAT", "SABCAT", predVars[-1])

sapply(regdf[predVars], function(x) levels(x)[1])   # the referent of each variable

# 1. Univariable: one logistic model per variable (each variable alone)
t0Zcrc <- uv_regression(regdf, predVars, outcome = "CANCER01")

# 2. Multivariable: all variables in one model
#    reformulate() writes the formula CANCER01 ~ AGECAT + RACE5CAT + ... from predVars
model0Zcrc <- glm(reformulate(predVars, response = "CANCER01"),
                  data = regdf,
                  family = "binomial")
summary(model0Zcrc)   # the raw model: coefficients are log-odds

t00Zcrc <- or_table(model0Zcrc, regdf, predVars)

# 3. Side by side (same variables and levels in the same order, so the rows line up)
table0Zcrc <- bind_cols(
    t0Zcrc  %>% rename(`Univariable OR (95% CI)` = OR, `Univariable p` = p),
    t00Zcrc %>% select(`Multivariable OR (95% CI)` = OR, `Multivariable p` = p))

table0Zcrc

# Part B · Coding lab

## Level 1

### Task 4.B · Your own Table 1

In [ ]:
# TASK 4.B: your own Table 1
# Example idea: among people with colorectal cancer, compare colon and rectal cancer.
cases <- analytic %>% filter(CANCERANY == "Colorectal cancer")

my_vars  <- c()          # TODO: 4 to 8 variables from names(analytic)
my_strat <- ""           # TODO: one grouping variable, e.g. "CANCERTYPE"

my_table <- table1(cases, vars = my_vars, by = my_strat)
my_table

In [ ]:
check("4.B", is.data.frame(my_table) && length(my_vars) >= 4 && all(my_vars %in% my_table$Variable),
      10, "Your Table 1 is built.", "Fill in my_vars (4 or more) and my_strat.")

### Task 4.C · Change the order of factor levels

Two tools change factor levels, and they do different things:

| Tool | What it does | Example |
| --- | --- | --- |
| `relevel(x, ref = "...")` | Moves **one** level to the front, making it the **referent**; the others keep their order | `relevel(EDU6CAT, ref = "College Degree")` |
| `factor(x, levels = c(...))` | Sets the **whole** order; the first level becomes the referent. List **every** level: a level you leave out becomes `NA` | `factor(INC6CAT, levels = c("150K or more", ..., "Missing income"))` |

The order changes how tables read and which group each odds ratio is compared with. It does **not** change the model's fit or the
p-value for each variable. After you refit, compare `my_or` with the multivariable columns of `table0Zcrc`: which odds ratios changed, and why?

In [ ]:
# TASK 4.C: change the order of factor levels, then refit the multivariable model
#   1. Education: make "College Degree" the referent with relevel()
#   2. Income: put the levels in a new order, highest first, with factor(levels = ...)
#      (the first level, "150K or more", becomes the referent)
my_regdf <- regdf %>%
    mutate(EDU6CAT = EDU6CAT,    # TODO: relevel(EDU6CAT, ref = ...)
           INC6CAT = INC6CAT)    # TODO: factor(INC6CAT, levels = c(...)), all six levels

levels(my_regdf$EDU6CAT)
levels(my_regdf$INC6CAT)

my_model <- glm(reformulate(predVars, response = "CANCER01"), data = my_regdf, family = "binomial")
my_or <- or_table(my_model, my_regdf, predVars)
my_or

In [ ]:
check("4.C", levels(my_regdf$EDU6CAT)[1] == "College Degree" &&
             identical(levels(my_regdf$INC6CAT), c("150K or more", "100-149K", "75-99K", "50-74K", "Less than 50K", "Missing income")) &&
             is.data.frame(my_or),
      10, "Referents changed and the model refitted.", "EDU6CAT should start with College Degree, and INC6CAT should list all six levels, highest first.")

### Task 4.D · Stratify Table 1 by different variables

**Stratifying** splits the table into columns, one per group of the `by` variable. Each column's percentages add up within
that group, and the p-value asks whether a variable's distribution differs across the groups. Changing `by` changes the question:

- `by = "CANCERANY"`: how do people with and without colorectal cancer differ?
- `by = "INS3CAT"`: how do insured and uninsured participants differ (including in how many have colorectal cancer)?
- `by = "BMICAT"` among cases: how does colorectal cancer differ across BMI groups?

The `by` variable can't also be in `vars`. More groups means smaller cells, so run `small_cells()` before you share anything.

In [ ]:
# TASK 4.D: Table 1 stratified by different variables
# 1. Everyone, by insurance (INS3CAT): include CANCERANY and 3 or more other variables
tab_ins <- NULL   # TODO: table1(analytic, vars = c(...), by = "INS3CAT")
tab_ins

# 2. People with colorectal cancer, by BMI group (BMICAT): include CANCERTYPE and 2 or more other variables
cases <- analytic %>% filter(CANCERANY == "Colorectal cancer")
tab_bmi <- NULL   # TODO: table1(cases, vars = c(...), by = "BMICAT")
tab_bmi

# Before sharing: any group from 1 to 19?
# small_cells(analytic, c(...), "INS3CAT")
# small_cells(cases, c(...), "BMICAT")

In [ ]:
check("4.D", is.data.frame(tab_ins) && all(c("Insurance", "No insurance") %in% names(tab_ins)) &&
             "CANCERANY" %in% tab_ins$Variable && sum(tab_ins$Variable != "" & tab_ins$Variable != "N") >= 4 &&
             is.data.frame(tab_bmi) && all(c("Normal", "Obese") %in% names(tab_bmi)) && "CANCERTYPE" %in% tab_bmi$Variable,
      10, "Two stratified tables built. Level 1 cleared!", "Build tab_ins by INS3CAT (with CANCERANY) and tab_bmi by BMICAT among cases (with CANCERTYPE).")

## Level 2 (boss level)

### Task 4.H · Under 40 vs 40–49 at first diagnosis

In [ ]:
# BOSS TASK 4.H: among people with cancer, Table 1 by age at first diagnosis (under 40 vs 40-49)
onset <- analytic %>%
    filter(CANCERANY == "Colorectal cancer") %>%
    mutate(ONSET = NA)   # TODO

onset_table <- NULL      # TODO: table1(onset, vars = ..., by = "ONSET")
onset_table
small_cells(onset, c("AGECAT", "RACE5CAT", "INS3CAT", "BMICAT"), "ONSET")

In [ ]:
check("4.H", is.data.frame(onset_table) && all(c("Under 40", "40-49") %in% names(onset_table)),
      20, "Boss defeated: onset Table 1 built.", "Make ONSET from AGEdxMINANY < 40 ('Under 40' or '40-49'), then table1(onset, vars = ..., by = 'ONSET').")

## Finish line

In [ ]:
scoreboard()
cat("\nReport your score (only your score) in the Zoom poll for Session 4.\n")